In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# --- Configuraciones Iniciales ---}
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-notifications")
ruta_driver = r"D:\Users\Lenovo\Documents\chrome-win\chromedriver.exe"
service = Service(ruta_driver)
driver = webdriver.Chrome(service=service, options=chrome_options)
url = "http://senasofiaplus.edu.co/sofia-public/"
driver.get(url)
time.sleep(3)

# Buscar y hacer clic en el botón de "Ingresar"
try:
    boton_ingresar = driver.find_element(By.XPATH, "//a[contains(text(), 'Ingresar')]")
    boton_ingresar.click()
    time.sleep(2)
except Exception as e:
    print(f"Error al encontrar el botón de Ingresar: {e}")

# Ingreso de credenciales
usuario = "1050962935"
contrasena = "PapaJose92805331050*"

def iniciar():
    """Función para el login."""
    try:
        driver.switch_to.default_content()
        driver.switch_to.frame("registradoBox1") 
        
        input_usuario = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "/html/body/form/div/div/div/div[2]/input")))
        input_usuario.send_keys(usuario)
        
        input_contrasena = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "/html/body/form/div/div/div/div[3]/input")))
        input_contrasena.send_keys(contrasena)
        
        boton_login = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "/html/body/form/div/div/div/div[7]/input")))
        boton_login.click()
        time.sleep(5)
        print("Inicio de sesión exitoso.")
    except Exception as e:
        print(f"Error al iniciar sesión: {e}")

# Intentar iniciar sesión
iniciar()

# --- NAVEGACIÓN ---

# 1. Seleccionar Rol (Option 4)
driver.switch_to.default_content() 
try:
    print("Seleccionando rol...")
    rol_xpath = '//*[@id="seleccionRol:roles"]/option[4]'
    WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, rol_xpath))).click()
    time.sleep(4)
except Exception as e:
    print(f"Error al seleccionar el rol: {e}")

# 2. Navegación al menú "Inscripción" y submenús
driver.switch_to.default_content()
try:
    print("Navegando al menú...")
    # Inscripción: //*[@id="side-menu"]/li[4]/a
    WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//*[@id="side-menu"]/li[4]/a'))).click()

    # Consultas: //*[@id="side-menu"]/li[4]/ul/li[1]/a
    WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//*[@id="side-menu"]/li[4]/ul/li[1]/a'))).click()

    # Generar reporte de inscripción: //*[@id="3230Opcion"]
    elemento_reporte = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, '//*[@id="3230Opcion"]')))
    driver.execute_script("arguments[0].click();", elemento_reporte)
    time.sleep(3)
    print("Llegada a 'Generar reporte de inscripción'.")

except Exception as e:
    print(f"Error durante la navegación del menú: {e}")


# 3. Solución para el clic en el icono de filtros (Punto de fallo)

# Cambiar al iframe "contenido"
try:
    driver.switch_to.default_content()
    WebDriverWait(driver, 10).until(EC.frame_to_be_available_and_switch_to_it((By.ID, "contenido")))
    print("Cambiado al iframe 'contenido'.")
    time.sleep(2)
except Exception as e:
    print(f"Error al cambiar al iframe 'contenido': {e}")
    # Si falla aquí, las siguientes líneas fallarán

# --- INTENTO MEJORADO DE CLIC EN EL ICONO ---

# El XPATH dinámico que falló: //*[@id="frmPrincipal:j_id_jsp_290781328_38"]
# Es mejor usar un XPATH basado en una clase o atributo más estático.

# Búsqueda basada en el ID (siempre es el primer intento)
icono_filtros_id = 'frmPrincipal:j_id_jsp_290781328_38' 

try:
    # 1. Intentar con el ID completo (Selenium a veces reemplaza los caracteres dinámicos)
    print("Intentando hacer clic en el icono de filtros (Método 1: ID)...")
    # Buscamos el elemento por el XPATH dinámico original si es el único identificador
    icono_filtros = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, '//*[@id="frmPrincipal:j_id_jsp_197265778_38"]'))
    )
    # Forzar el clic con JavaScript
    driver.execute_script("arguments[0].click();", icono_filtros)
    print("Clic en icono de filtros ejecutado con éxito.")
    time.sleep(2)
    
except Exception as e:
    print(f"Fallo el Metodo 1: {e}. Intentando Metodo 2 (XPATH con clase)...")
    
    # 2. Intentar con XPATH que busca el icono de lupa (ui-icon-search)
    # Este XPATH busca el span que generalmente representa el icono de búsqueda/filtro
    icono_filtros_clase = '/html/body[1]/div[2]/form/div[1]/fieldset/table/tbody/tr[1]/td[2]/table/tbody/tr/td[2]/a/img'
    try:
        icono_filtros = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, icono_filtros_clase))
        )
        driver.execute_script("arguments[0].click();", icono_filtros)
        print("Clic en icono de filtros ejecutado con éxito (Método 2).")
        time.sleep(2)
    except Exception as e:
        print(f"Fallo el Metodo 2. Error final al hacer clic en el icono de filtros: {e}")
        
    

# 4. Escribir la ficha en el input
ficha_a_ingresar = "123456"
input_ficha_xpath = '//*[@id="form:codigoFichaITX"]'

IFRAME_MODAL_ID = 'modalDialogContentviewDialog2'

try:
    print("⏳ Esperando iframe 'modal'...")
    iframe_element = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, IFRAME_MODAL_ID))
    )
    driver.switch_to.frame(iframe_element)
    print("✅ Iframe 'modal' activado.")

    print(f"Escribiendo la ficha {ficha_a_ingresar} en el input...")
    # Esperar a que el input exista dentro del iframe ya activo
    input_ficha = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, input_ficha_xpath))
    )
    input_ficha.clear()
    input_ficha.send_keys(ficha_a_ingresar)
    print("Ficha ingresada con éxito.")

except Exception as e:
    print(f"❌ Error al interactuar con el modal o el input de la ficha: {e}")

# Opcional: Mantener el navegador abierto (descomenta si lo necesitas)
# input("Presiona Enter para cerrar el navegador...") 
# driver.quit()

Error al encontrar el botón de Ingresar: Message: no such element: Unable to locate element: {"method":"xpath","selector":"//a[contains(text(), 'Ingresar')]"}
  (Session info: chrome=103.0.5046.0); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
Backtrace:
	Ordinal0 [0x01096903+2844931]
	Ordinal0 [0x00F74271+1655409]
	Ordinal0 [0x00E0013A+131386]
	Ordinal0 [0x00E28293+295571]
	Ordinal0 [0x00E28412+295954]
	Ordinal0 [0x00E4BAF2+441074]
	Ordinal0 [0x00E3C5B4+378292]
	Ordinal0 [0x00E4A2DE+434910]
	Ordinal0 [0x00E3C2E6+377574]
	Ordinal0 [0x00E1F2CB+258763]
	Ordinal0 [0x00E200B6+262326]
	Ordinal0 [0x0109A9B5+2861493]
	GetHandleVerifier [0x012156CD+1448573]
	GetHandleVerifier [0x0121531A+1447626]
	GetHandleVerifier [0x01220E46+1495542]
	GetHandleVerifier [0x01215E74+1450532]
	Ordinal0 [0x00F86F3B+1732411]
	Ordinal0 [0x00F93058+1781848]
	Ordinal0 [0x00F931FE+1782270]
	Ordinal0 [0x00FA6